# 玉山銀行警示帳戶交易異常檢測 Notebook

說明：此 Notebook 已拆成多個 cell（方便逐段執行與除錯），每段程式或每個函式均已加入繁體中文註解。

使用步驟：
1. 下載本檔案並以 Jupyter Notebook / JupyterLab 開啟（請勿把 JSON 貼到單一 cell 執行）。
2. 將 alert_transaction.csv 與 acct_alert.csv 放在相同資料夾或修改 TXN_CSV、ALERT_ACCT_CSV 變數。
3. 逐個 cell 執行（遇錯誤時只需重新執行該 cell 或受影響的下游 cell）。


In [1]:
# Cell 1: 匯入套件與基本設定（含繁體中文註解）
import os
import math
import time
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import font_manager

sns.set(style='whitegrid')

# 圖檔與細節輸出資料夾
OUT_DIR = 'notebook_details'
os.makedirs(OUT_DIR, exist_ok=True)

# 參數：檔案位置與偵測參數（可視需要調整）
TXN_CSV = 'alert_transaction.csv'
ALERT_ACCT_CSV = 'acct_alert.csv'
BASE_DATE = '2020-01-01'
SHORT_WINDOW_HOURS = 2
SAME_AMT_WINDOW_DAYS = 1
SMURF_WINDOW_DAYS = 1
HUB_LOOKBACK_DAYS = 7
SHORT_MULTI_MIN_COUNT = 3
SHORT_MULTI_MIN_UNIQ_SOURCES = 3
SAME_AMT_MIN_SOURCES = 3
SMURF_AMT_THRESHOLD = 1000
SMURF_COUNT_THRESHOLD = 10
HUB_UNIQUE_SOURCE_THRESHOLD = 12
HUB_OUTFLOW_FRACTION = 0.6
TIME_CLUSTER_COUNT = 3

RISK_WEIGHTS = {
    'short_multi_in': 4.0,
    'same_amt_multi_src': 3.0,
    'smurfing_small': 1.0,
    'hub_and_spoke': 4.5,
    'time_cluster': 1.5,
    'currency_mix': 1.2,
    'high_degree': 2.0,
    'high_entropy_counterparty': 1.5
}


In [2]:
# Cell 2: 字型設定（中文註解）
# 目的：確保 matplotlib 在互動顯示與 savefig 時都能使用支援中文的字型。
preferred_fonts = ['Microsoft JhengHei', 'SimHei', 'DejaVu Sans']
found_font = None
for fname in preferred_fonts:
    try:
        path = font_manager.findfont(fname, fallback_to_default=False)
        # 當找到系統字型時，註冊到 font manager 並設為主要字型
        font_manager.fontManager.addfont(path)
        plt.rcParams['font.family'] = fname
        found_font = fname
        break
    except Exception:
        continue

if found_font is None:
    # 若系統找不到上述字型，將 preferred_fonts 設為 sans-serif 名單
    plt.rcParams['font.sans-serif'] = preferred_fonts
    plt.rcParams['font.family'] = 'sans-serif'

plt.rcParams['axes.unicode_minus'] = False
print('設定字型完成。使用字型：', found_font if found_font else plt.rcParams.get('font.sans-serif'))

# 註：若 savefig 時仍出現亂碼，請把系統字型檔案路徑（例如 C:\\Windows\\Fonts\\msjh.ttf）用 font_manager.addfont() 明確加入，並重啟 kernel。

設定字型完成。使用字型： Microsoft JhengHei


In [3]:
# Cell 3: 嘗試安全載入 networkx（若失敗則跳過網路視覺化）
try:
    import networkx as nx
    NX_AVAILABLE = True
    print('networkx 可用，版本：', nx.__version__)
except Exception as e:
    NX_AVAILABLE = False
    nx = None
    print('警告：networkx 無法載入，網路圖功能將被略過。錯誤：', e)

# 匯入 Jupyter display 以便後續使用 display()
try:
    from IPython.display import display
except Exception:
    # fallback: 定義簡單的 display 函數
    def display(df):
        try:
            print(df.head().to_string())
        except Exception:
            print(df)


networkx 可用，版本： 3.5


In [4]:
# Cell 4: 載入資料並建立 dt 欄位（含中文註解）
def load_and_prepare(txn_path=TXN_CSV, acct_alert_path=ALERT_ACCT_CSV):
    """
    讀入 CSV 並做基本欄位轉換：
    - txn_amt 轉成數值
    - txn_date 轉成整數
    - txn_time 若為空則設為 00:00:00
    - 建立 dt 欄位（以 BASE_DATE + txn_date offset + txn_time）
    回傳 tx (DataFrame) 與 acct_alert (DataFrame)
    """
    tx = pd.read_csv(txn_path, dtype=str).fillna("")
    tx['txn_amt'] = pd.to_numeric(tx['txn_amt'], errors='coerce').fillna(0.0)
    tx['txn_date'] = pd.to_numeric(tx['txn_date'], errors='coerce').fillna(0).astype(int)
    tx['txn_time'] = tx['txn_time'].replace('', '00:00:00')
    tx['channel_type'] = tx['channel_type'].replace('', 'UNK')

    # 安全解析時間字串，避免格式例外中斷
    def safe_timedelta(t):
        try:
            return pd.to_timedelta(t)
        except Exception:
            return pd.to_timedelta('00:00:00')
    tx['time_only'] = tx['txn_time'].apply(safe_timedelta)
    base = pd.to_datetime(BASE_DATE)
    tx['dt'] = base + pd.to_timedelta(tx['txn_date'] - 1, unit='D') + tx['time_only']
    acct_alert = pd.read_csv(acct_alert_path, dtype=str).fillna("")
    return tx, acct_alert

# 執行讀取並列印數量（這兩行會在 cell 執行時直接顯示）
tx, acct_alert = load_and_prepare()
print('交易筆數：', len(tx))
print('警示帳戶數：', len(acct_alert))


交易筆數： 33933
警示帳戶數： 1004


In [6]:
# Cell 5: 取得單一帳戶的交易記錄（方向與對手欄位）
def get_acct_txns(tx, acct):
    """
    取得某個帳戶的所有交易（包含 from 與 to），
    並加上 direction 與 counterparty 欄位：
    - direction: 'in' 或 'out'
    - counterparty: 對手帳號
    """
    mask = (tx['from_acct'] == acct) | (tx['to_acct'] == acct)
    df = tx.loc[mask].copy()
    if df.empty:
        return df
    df['direction'] = np.where(df['to_acct'] == acct, 'in', 'out')
    df['counterparty'] = np.where(df['direction'] == 'in', df['from_acct'], df['to_acct'])
    df = df.sort_values('dt').reset_index(drop=True)
    return df

# 範例：顯示第一個警示帳戶的交易筆數（若有警示帳戶）
if len(acct_alert) > 0:
    sample_acct = acct_alert.loc[0, 'acct']
    print('示例帳戶：', sample_acct, '交易筆數：', len(get_acct_txns(tx, sample_acct)))


示例帳戶： 80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d6b507091e36e25d279 交易筆數： 8


In [7]:
# Cell 6: 特徵工程函數（每個欄位都有中文註解）
from scipy.stats import entropy

def acct_features(df_acct):
    """
    計算帳戶級別特徵：交易數、金額統計、對手數、entropy、time-based 等
    回傳一個字典（key 為特徵名稱）
    """
    if df_acct.empty:
        return {}
    feat = {}
    # 基本計數
    feat['total_txns'] = len(df_acct)
    feat['in_txns'] = int((df_acct['direction']=='in').sum())
    feat['out_txns'] = int((df_acct['direction']=='out').sum())
    # 金額相關
    feat['sum_in_amt'] = float(df_acct[df_acct['direction']=='in']['txn_amt'].sum())
    feat['sum_out_amt'] = float(df_acct[df_acct['direction']=='out']['txn_amt'].sum())
    feat['max_txn_amt'] = float(df_acct['txn_amt'].max())
    feat['mean_txn_amt'] = float(df_acct['txn_amt'].mean())
    feat['std_txn_amt'] = float(df_acct['txn_amt'].std(ddof=0) if feat['total_txns']>1 else 0.0)
    # 重複金額
    amt_counts = df_acct['txn_amt'].value_counts()
    feat['top_amount'] = float(amt_counts.index[0]) if not amt_counts.empty else 0.0
    feat['top_amount_count'] = int(amt_counts.iloc[0]) if not amt_counts.empty else 0
    # 對手分佈
    cp_counts = df_acct['counterparty'].value_counts()
    feat['unique_counterparties'] = int(cp_counts.size)
    # top recipient fraction（出款主要目的地比例）
    out = df_acct[df_acct['direction']=='out']
    if not out.empty:
        top_recipient_amt = out.groupby('counterparty')['txn_amt'].sum().max()
        feat['top_recipient_frac'] = float(top_recipient_amt / out['txn_amt'].sum())
    else:
        feat['top_recipient_frac'] = 0.0
    # entropy（用來衡量對手分佈均勻度）
    counts = cp_counts.values
    feat['cp_entropy'] = float(entropy(counts, base=2)) if counts.sum()>0 else 0.0
    # degree-like（不同來源/目的帳號數）
    feat['in_degree'] = int(df_acct[df_acct['direction']=='in']['counterparty'].nunique())
    feat['out_degree'] = int(df_acct[df_acct['direction']=='out']['counterparty'].nunique())
    # 時間間隔特徵
    times = df_acct['dt'].sort_values()
    time_diffs = times.diff().dropna().dt.total_seconds()
    feat['mean_interarrival_sec'] = float(time_diffs.mean()) if not time_diffs.empty else None
    feat['median_interarrival_sec'] = float(time_diffs.median()) if not time_diffs.empty else None
    # 24小時與 2小時內的交易數
    t_end = df_acct['dt'].max()
    feat['count_24h'] = int(((df_acct['dt'] >= (t_end - pd.Timedelta(days=1))) & (df_acct['dt'] <= t_end)).sum())
    feat['count_2h'] = int(((df_acct['dt'] >= (t_end - pd.Timedelta(hours=SHORT_WINDOW_HOURS))) & (df_acct['dt'] <= t_end)).sum())
    # 小額計數（smurf 判斷用）
    feat['count_small_1000'] = int((df_acct['txn_amt'] <= SMURF_AMT_THRESHOLD).sum())
    # 幣別與通路
    feat['num_currency_types'] = int(df_acct['currency_type'].nunique())
    feat['dominant_currency_pct'] = float(df_acct['currency_type'].value_counts(normalize=True).iloc[0]) if df_acct['currency_type'].nunique()>0 else 0.0
    feat['num_channel_types'] = int(df_acct['channel_type'].nunique())
    feat['pct_channel_unk'] = float((df_acct['channel_type']=='UNK').mean())
    # 同金額多來源（24h）簡易統計
    in_df = df_acct[df_acct['direction']=='in']
    feat['same_amt_multi_src_24h'] = 0
    if not in_df.empty:
        grp = in_df.groupby('txn_amt')['counterparty'].nunique()
        feat['same_amt_multi_src_24h'] = int((grp >= SAME_AMT_MIN_SOURCES).sum())
    return feat


In [8]:
# Cell 7: 規則檢測函數（每個函式都加上中文註解）
def detect_short_window_multi_inbound(df_acct, window_hours=SHORT_WINDOW_HOURS, min_count=SHORT_MULTI_MIN_COUNT, min_unique_sources=SHORT_MULTI_MIN_UNIQ_SOURCES):
    """
    檢測：在短時窗 (window_hours) 內是否有 >= min_count 筆入款，且來源帳號數 >= min_unique_sources
    回傳 (flag, detail_list)，detail_list 為觸發視窗的交易片段
    """
    in_df = df_acct[df_acct['direction']=='in'].copy()
    if in_df.empty:
        return False, []
    trig = []
    window = pd.Timedelta(hours=window_hours)
    for idx,row in in_df.iterrows():
        t_end = row['dt']
        t_start = t_end - window
        w = in_df[(in_df['dt'] >= t_start) & (in_df['dt'] <= t_end)]
        if len(w) >= min_count and w['counterparty'].nunique() >= min_unique_sources:
            trig.append({'start':t_start, 'end':t_end, 'count':len(w), 'unique_sources':w['counterparty'].nunique(), 'txns': w.copy()})
    return (len(trig)>0), trig

def detect_same_amount_multiple_sources(df_acct, window_days=SAME_AMT_WINDOW_DAYS, min_sources=SAME_AMT_MIN_SOURCES):
    """
    檢測：在 window_days 內同金額由 >= min_sources 個不同來源匯入
    回傳 (flag, detail_list)
    """
    in_df = df_acct[df_acct['direction']=='in'].copy()
    if in_df.empty:
        return False, []
    trig = []
    for idx,row in in_df.iterrows():
        t_end = row['dt']
        t_start = t_end - pd.Timedelta(days=window_days)
        w = in_df[(in_df['dt'] >= t_start) & (in_df['dt'] <= t_end)]
        if w.empty: continue
        grp = w.groupby('txn_amt')['counterparty'].nunique()
        candidate = grp[grp >= min_sources]
        for amt, uniq_src in candidate.items():
            txns = w[w['txn_amt'] == amt].copy()
            trig.append({'start':t_start, 'end':t_end, 'txn_amt':float(amt), 'unique_sources':int(uniq_src), 'txns': txns})
    return (len(trig)>0), trig

def detect_smurfing_small(df_acct, amount_thr=SMURF_AMT_THRESHOLD, count_thr=SMURF_COUNT_THRESHOLD, window_days=SMURF_WINDOW_DAYS):
    """
    檢測：在 window_days 內小於 amount_thr 的交易數是否 >= count_thr
    回傳 (flag, detail_list)
    """
    df = df_acct.copy()
    small = df[df['txn_amt'] <= amount_thr]
    if small.empty:
        return False, []
    trig = []
    for idx,row in small.iterrows():
        t_end = row['dt']
        t_start = t_end - pd.Timedelta(days=window_days)
        w = small[(small['dt'] >= t_start) & (small['dt'] <= t_end)]
        if len(w) >= count_thr:
            trig.append({'start':t_start, 'end':t_end, 'count_small':len(w), 'txns': w.copy()})
    return (len(trig)>0), trig

def detect_hub_and_spoke(df_acct, lookback_days=HUB_LOOKBACK_DAYS, unique_src_thr=HUB_UNIQUE_SOURCE_THRESHOLD, outflow_frac_thr=HUB_OUTFLOW_FRACTION):
    """
    檢測 hub-and-spoke：在 lookback_days 內，unique_in_sources 是否大於門檻，且 outflow 到 top recipient 比例高
    回傳 (flag, details_dict)
    """
    if df_acct.empty:
        return False, None
    t_end = df_acct['dt'].max()
    t_start = t_end - pd.Timedelta(days=lookback_days)
    w = df_acct[(df_acct['dt'] >= t_start) & (df_acct['dt'] <= t_end)]
    in_df = w[w['direction']=='in']
    out_df = w[w['direction']=='out']
    unique_in = in_df['counterparty'].nunique()
    total_out = out_df['txn_amt'].sum()
    top_rec_amt = out_df.groupby('counterparty')['txn_amt'].sum().max() if not out_df.empty else 0.0
    top_frac = float(top_rec_amt / total_out) if total_out>0 else 0.0
    ok = (unique_in >= unique_src_thr) and (top_frac >= outflow_frac_thr)
    details = {'start':t_start, 'end':t_end, 'unique_in':int(unique_in), 'total_out':float(total_out), 'top_rec_amt':float(top_rec_amt), 'top_frac':float(top_frac), 'in_txns':in_df.copy(), 'out_txns':out_df.copy()}
    return ok, details

def detect_time_cluster(df_acct, same_minute_count=TIME_CLUSTER_COUNT):
    """
    檢測：是否有同一分鐘內來自多個不同來源的入款（>= same_minute_count）
    """
    in_df = df_acct[df_acct['direction']=='in'].copy()
    if in_df.empty:
        return False, []
    in_df['minute'] = in_df['dt'].dt.floor('T')
    grp = in_df.groupby('minute')['counterparty'].nunique().reset_index(name='unique_sources')
    triggered = grp[grp['unique_sources'] >= same_minute_count]
    out = []
    for _, r in triggered.iterrows():
        minute = r['minute']
        txns = in_df[in_df['minute']==minute].copy()
        out.append({'minute': minute, 'unique_sources': int(r['unique_sources']), 'txns': txns})
    return (len(out)>0), out

def detect_currency_mix(df_acct, window_hours=1):
    """
    檢測：短時窗內是否同時出現多種幣別（如 TWD 與 USD）
    """
    if df_acct.empty:
        return False, []
    trigs = []
    for idx,row in df_acct.iterrows():
        t_end = row['dt']
        t_start = t_end - pd.Timedelta(hours=window_hours)
        w = df_acct[(df_acct['dt'] >= t_start) & (df_acct['dt'] <= t_end)]
        if w['currency_type'].nunique() >= 2:
            trigs.append({'start':t_start, 'end':t_end, 'currencies': list(w['currency_type'].unique()), 'txns': w.copy()})
    return (len(trigs)>0), trigs


In [9]:
# Cell 8: 風險打分函數（將以上規則合併並回傳 score 與觸發清單）
def score_account(df_acct, feats=None):
    """
    將多項檢測結果依權重加總，回傳 risk_score 與 triggered list，以及各檢測的 detail
    """
    if df_acct.empty:
        return {'risk_score': 0.0, 'triggered': []}
    if feats is None:
        feats = acct_features(df_acct)
    score = 0.0
    triggered = []
    ok, det = detect_short_window_multi_inbound(df_acct)
    if ok:
        score += RISK_WEIGHTS['short_multi_in']; triggered.append('short_multi_in')
    ok, det2 = detect_same_amount_multiple_sources(df_acct)
    if ok:
        score += RISK_WEIGHTS['same_amt_multi_src']; triggered.append('same_amt_multi_src')
    ok, det3 = detect_smurfing_small(df_acct)
    if ok:
        score += RISK_WEIGHTS['smurfing_small']; triggered.append('smurfing_small')
    ok, det4 = detect_hub_and_spoke(df_acct)
    if ok:
        score += RISK_WEIGHTS['hub_and_spoke']; triggered.append('hub_and_spoke')
    ok, det5 = detect_time_cluster(df_acct)
    if ok:
        score += RISK_WEIGHTS['time_cluster']; triggered.append('time_cluster')
    ok, det6 = detect_currency_mix(df_acct)
    if ok:
        score += RISK_WEIGHTS['currency_mix']; triggered.append('currency_mix')
    if feats.get('in_degree',0) + feats.get('out_degree',0) >= 20:
        score += RISK_WEIGHTS['high_degree']; triggered.append('high_degree')
    if feats.get('cp_entropy',0) >= 4.0:
        score += RISK_WEIGHTS['high_entropy_counterparty']; triggered.append('high_entropy_counterparty')
    return {'risk_score': score, 'triggered': triggered, 'det_short_multi': det, 'det_same_amt': det2, 'det_smurf': det3, 'det_hub': det4, 'det_timecluster': det5, 'det_currency': det6}


In [12]:
# 新增一個 cell：定義 save_fig（中文註解）
# 請在執行視覺化 cell 前執行此 cell（放在 notebook 的第 1~9 cell 之後，第 10 cell 之前）
import os
from datetime import datetime

# 確保 OUT_DIR 存在（若你已在其他 cell 定義 OUT_DIR 可刪除這行）
OUT_DIR = globals().get('OUT_DIR', 'notebook_details')
os.makedirs(OUT_DIR, exist_ok=True)

def save_fig(fig=None, name_prefix='figure', out_dir=OUT_DIR, dpi=150):
    """
    儲存 matplotlib figure 的輔助函式（中文註解）
    - fig: matplotlib.figure.Figure 物件；若為 None，會取目前的 plt.gcf()
    - name_prefix: 檔名開頭
    - out_dir: 輸出資料夾
    - 回傳：儲存檔案的完整路徑
    """
    import matplotlib.pyplot as _plt
    if fig is None:
        fig = _plt.gcf()
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    safe_name = f"{name_prefix}_{ts}.png"
    path = os.path.join(out_dir, safe_name)
    try:
        # 嘗試儲存並通知路徑
        fig.savefig(path, bbox_inches='tight', dpi=dpi)
        print(f"已儲存圖檔: {path}")
    except Exception as e:
        print("儲存圖檔失敗：", e)
    try:
        _plt.close(fig)
    except Exception:
        pass
    return path

# 測試（可選）：若你已有一張圖可顯示，可呼叫 save_fig() 測試
# 例如：
# import matplotlib.pyplot as plt
# plt.plot([1,2,3],[1,4,9]); p = save_fig(None, 'test_plot')

In [13]:
# Cell 9: 安靜模式分析所有帳戶（僅顯示開始與完成訊息）
def analyze_all_quiet(tx, acct_alert, out_dir=OUT_DIR):
    """
    逐一分析 acct_alert 中的帳戶，並把每個帳戶的交易細節與觸發視窗存成 CSV。
    不會在每筆帳戶處理時列印大量訊息（避免 Notebook Output 爆掉）。
    執行完後會產生 acct_risk_summary.csv 與各帳戶資料夾。
    """
    rows = []
    acct_list = acct_alert['acct'].tolist()
    total = len(acct_list)
    print('帳戶異常交易分析儲存中：共', total, '筆帳戶，請稍候...')
    for acct in acct_list:
        r = acct_alert.loc[acct_alert['acct']==acct].iloc[0]
        event_date = r.get('event_date','')
        df_acct = get_acct_txns(tx, acct)
        if df_acct.empty:
            rows.append({'acct': acct, 'event_date': event_date, 'total_txns': 0, 'risk_score': 0.0, 'triggered_rules': ''})
            continue
        feats = acct_features(df_acct)
        sc = score_account(df_acct, feats)
        # 儲存帳戶交易與觸發視窗 CSV
        acct_dir = os.path.join(out_dir, acct)
        os.makedirs(acct_dir, exist_ok=True)
        df_acct.to_csv(os.path.join(acct_dir, f"{acct}_txns.csv"), index=False)
        # 若觸發規則則分別存檔（detail 會放在對應帳戶資料夾）
        if sc.get('det_short_multi'):
            for idx,w in enumerate(sc['det_short_multi']):
                w['txns'].to_csv(os.path.join(acct_dir, f"short_multi_{idx}.csv"), index=False)
        if sc.get('det_same_amt'):
            for idx,w in enumerate(sc['det_same_amt']):
                w['txns'].to_csv(os.path.join(acct_dir, f"same_amt_{idx}.csv"), index=False)
        if sc.get('det_smurf'):
            for idx,w in enumerate(sc['det_smurf']):
                w['txns'].to_csv(os.path.join(acct_dir, f"smurf_{idx}.csv"), index=False)
        if sc.get('det_hub') and isinstance(sc['det_hub'], dict):
            sc['det_hub']['in_txns'].to_csv(os.path.join(acct_dir, "hub_in_txns.csv"), index=False)
            sc['det_hub']['out_txns'].to_csv(os.path.join(acct_dir, "hub_out_txns.csv"), index=False)
        if sc.get('det_timecluster'):
            for idx,w in enumerate(sc['det_timecluster']):
                w['txns'].to_csv(os.path.join(acct_dir, f"time_cluster_{idx}.csv"), index=False)
        if sc.get('det_currency'):
            for idx,w in enumerate(sc['det_currency']):
                w['txns'].to_csv(os.path.join(acct_dir, f"currency_mix_{idx}.csv"), index=False)
        # 將帳戶特徵與風險結果收集到 summary
        rows.append({
            'acct': acct,
            'event_date': event_date,
            'total_txns': feats.get('total_txns',0),
            'in_txns': feats.get('in_txns',0),
            'out_txns': feats.get('out_txns',0),
            'sum_in_amt': feats.get('sum_in_amt',0.0),
            'sum_out_amt': feats.get('sum_out_amt',0.0),
            'in_degree': feats.get('in_degree',0),
            'out_degree': feats.get('out_degree',0),
            'cp_entropy': feats.get('cp_entropy',0.0),
            'top_amount': feats.get('top_amount',0.0),
            'top_amount_count': feats.get('top_amount_count',0),
            'count_24h': feats.get('count_24h',0),
            'count_2h': feats.get('count_2h',0),
            'count_small_1000': feats.get('count_small_1000',0),
            'num_currency_types': feats.get('num_currency_types',0),
            'pct_channel_unk': feats.get('pct_channel_unk',0.0),
            'risk_score': sc['risk_score'],
            'triggered_rules': "|".join(sc['triggered'])
        })
    df_summary = pd.DataFrame(rows).sort_values('risk_score', ascending=False).reset_index(drop=True)
    df_summary.to_csv(os.path.join(out_dir, "acct_risk_summary.csv"), index=False)
    print('分析完成。')
    return df_summary

# 執行分析：執行此 cell 時會先顯示開始訊息，完成後印出 summary 的位置
summary_df = analyze_all_quiet(tx, acct_alert)
print('Summary 存於:', os.path.join(OUT_DIR, 'acct_risk_summary.csv'))
print('圖檔與各帳戶細節存於資料夾:', OUT_DIR)


帳戶異常交易分析儲存中：共 1004 筆帳戶，請稍候...
分析完成。
Summary 存於: notebook_details\acct_risk_summary.csv
圖檔與各帳戶細節存於資料夾: notebook_details


In [14]:
# Cell 10: 視覺化（直方圖、Top N 條狀圖、每日入出）
fig = plt.figure(figsize=(10,5))
sns.histplot(summary_df['risk_score'], bins=20, kde=False)
plt.title('帳戶風險分數分佈')
plt.xlabel('risk_score')
plt.ylabel('帳戶數')
hist_path = save_fig(fig, 'risk_score_distribution')

TOP_N = 20
topN = summary_df.head(TOP_N)
fig = plt.figure(figsize=(12,6))
sns.barplot(x='risk_score', y='acct', data=topN, palette='Reds_r')
plt.title(f'Top {TOP_N} 高風險帳戶')
plt.xlabel('risk_score')
plt.ylabel('acct')
bar_path = save_fig(fig, 'top_risk_accounts')

if not topN.empty:
    top_acct = topN.iloc[0]['acct']
    df_top = get_acct_txns(tx, top_acct).copy()
    if not df_top.empty:
        df_top['date'] = df_top['dt'].dt.floor('D')
        daily = df_top.groupby(['date','direction'])['txn_amt'].sum().unstack(fill_value=0)
        fig = daily.plot(kind='bar', stacked=False, figsize=(12,6)).get_figure()
        plt.title(f'{top_acct} 每日入/出金額')
        plt.ylabel('金額')
        daily_path = save_fig(fig, f'{top_acct}_daily_in_out')
        print('已產生 Top 圖檔與每日入出圖檔。')


已儲存圖檔: notebook_details\risk_score_distribution_20251030_113921.png


C:\Users\macrj\AppData\Local\Temp\ipykernel_35428\2276767719.py:12: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='risk_score', y='acct', data=topN, palette='Reds_r')


已儲存圖檔: notebook_details\top_risk_accounts_20251030_113921.png
已儲存圖檔: notebook_details\762492f9501d32b495086ee115bbfb1d866cc1d93e6ff6c1d3418fc91251dfab_daily_in_out_20251030_113921.png
已產生 Top 圖檔與每日入出圖檔。


In [15]:
# Cell 11: 網路圖（若 networkx 可用則畫網路並存檔）
def build_flow_graph(tx, accts, last_days=30):
    """
    建立交易網路圖：節點為帳號，邊重為交易金額加總
    """
    t_end = tx['dt'].max()
    t_start = t_end - pd.Timedelta(days=last_days)
    sub = tx[(tx['dt'] >= t_start) & (tx['dt'] <= t_end)]
    mask = sub['from_acct'].isin(accts) | sub['to_acct'].isin(accts)
    sub = sub[mask]
    G = nx.DiGraph()
    for _, row in sub.iterrows():
        u = row['from_acct']
        v = row['to_acct']
        w = float(row['txn_amt'])
        if G.has_edge(u,v):
            G[u][v]['weight'] += w
            G[u][v]['count'] += 1
        else:
            G.add_edge(u,v, weight=w, count=1)
    return G

if NX_AVAILABLE:
    top_accounts = summary_df.head(10)['acct'].tolist()
    G = build_flow_graph(tx, top_accounts, last_days=60)
    print('節點數:', G.number_of_nodes(), '邊數:', G.number_of_edges())
    if G.number_of_nodes() > 0:
        fig = plt.figure(figsize=(12,12))
        pos = nx.spring_layout(G, k=0.3, iterations=100)
        degrees = dict(G.degree())
        node_sizes = [max(80, degrees.get(n,1)*80) for n in G.nodes()]
        edge_widths = [math.log1p(d['weight']) for _,_,d in G.edges(data=True)]
        nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color='orange', alpha=0.9)
        nx.draw_networkx_edges(G, pos, arrowstyle='->', arrowsize=10, width=edge_widths, alpha=0.6)
        labels = {n: (n if n in top_accounts else '') for n in G.nodes()}
        nx.draw_networkx_labels(G, pos, labels, font_size=8)
        plt.title('Top 帳戶交易網路 (節點=帳號, 邊寬~金額log)')
        plt.axis('off')
        graph_path = save_fig(fig, 'top_accounts_network')
else:
    print('跳過網路圖（networkx 無法使用）')


節點數: 641 邊數: 653
已儲存圖檔: notebook_details\top_accounts_network_20251030_114004.png


## 完成
- summary CSV： `notebook_details/acct_risk_summary.csv`
- 圖檔與每個帳戶的 detail CSV：都在 `notebook_details/` 資料夾

請逐個 cell 執行（依序執行 Cell 1 ~ Cell 11）。若要我把此 .ipynb 直接放到你提供的 GitHub repo，請提供 repo 與 branch 資訊與授權，我可以幫你提交。

In [20]:
# 篩選負例並輸出 acct_normal.csv 與 normal_transaction.csv
# 把這個 cell 放在你 notebook 中，建議位置：在已產生 acct_alert 的 cell 之後、在做訓練前執行。
# 參數可以修改：MASTER_TXN_CSV, NEG_SAMPLE_N, SAMPLE_SOURCE, MODE
import os
import pandas as pd
import random
from collections import Counter

# ------------- 可調參數 -------------
MASTER_TXN_CSV = "acct_transaction.csv"   # 母集 CSV（請確定在 notebook 工作目錄）
OUT_ACCT_NORMAL = "acct_normal.csv"
OUT_NORMAL_TXN = "normal_transaction.csv"
NEG_SAMPLE_N = None        # None => 與正例數量平衡；或指定整數 e.g. 1004
SAMPLE_SOURCE = 'random'   # 'random' | 'match_activity' | 'from_predict'
MODE = 'chunk'             # 'full' 或 'chunk'（建議 chunk）
CHUNK_SIZE = 200000
RANDOM_SEED = 42
# -------------------------------------

random.seed(RANDOM_SEED)

# 1) 取得正例 acct_alert（優先使用 kernel 中的變數）
pos_accts = None
if 'acct_alert' in globals() and isinstance(acct_alert, pd.DataFrame) and 'acct' in acct_alert.columns:
    pos_accts = acct_alert['acct'].astype(str).unique().tolist()
    print(f"[info] 使用 kernel 中的 acct_alert，共 {len(pos_accts)} 筆正例。")
else:
    # 若沒有，嘗試讀檔案
    if os.path.exists("acct_alert.csv"):
        tmp = pd.read_csv("acct_alert.csv", dtype=str).fillna("")
        if 'acct' in tmp.columns:
            pos_accts = tmp['acct'].astype(str).unique().tolist()
            print(f"[info] 從 acct_alert.csv 讀入正例，共 {len(pos_accts)} 筆。")

if pos_accts is None:
    raise RuntimeError("找不到正例清單 (acct_alert)。請先在 kernel 中載入 acct_alert 或放置 acct_alert.csv。")

pos_n = len(pos_accts)

# 2) 決定負例數量
if NEG_SAMPLE_N is None:
    neg_n = pos_n
else:
    neg_n = int(NEG_SAMPLE_N)
print(f"[info] 目標負例數 neg_n = {neg_n}")

# 3) 取得母集中的所有帳戶（逐 chunk 掃描以節省記憶體）
def iter_master_accounts(master_csv, chunksize=CHUNK_SIZE):
    for chunk in pd.read_csv(master_csv, dtype=str, chunksize=chunksize):
        chunk = chunk.fillna("")
        for acct in pd.unique(chunk[['from_acct','to_acct']].values.ravel()):
            if acct != "" and acct is not None:
                yield acct

print("[info] 開始掃描母集以蒐集候選帳戶（此動作會逐 chunk 讀取檔案，視檔案大小可能需要數十秒至數分鐘）...")
candidate_set = set()
if MODE == 'full':
    # 警告：full 模式會一次性載入整個檔案，僅在本機記憶體充足時使用
    dfm = pd.read_csv(MASTER_TXN_CSV, dtype=str).fillna("")
    candidate_set.update(dfm['from_acct'].astype(str).unique().tolist())
    candidate_set.update(dfm['to_acct'].astype(str).unique().tolist())
else:
    # chunk 模式（推薦）
    for acct in iter_master_accounts(MASTER_TXN_CSV, CHUNK_SIZE):
        candidate_set.add(str(acct))

print(f"[info] 母集帳戶總數（包含正例）: {len(candidate_set)}")

# 排除正例
pos_set = set(map(str, pos_accts))
neg_candidates = [a for a in candidate_set if a not in pos_set]
print(f"[info] 去除正例後的負例候選數: {len(neg_candidates)}")

if len(neg_candidates) == 0:
    raise RuntimeError("負例候選為空。請檢查母集與正例清單。")

# 4) 抽樣（簡單隨機或 match by activity）
if SAMPLE_SOURCE == 'random' or SAMPLE_SOURCE == 'from_predict':
    # 若 from_predict, user 需要先載入 acct_predict.csv 且將 SAMPLE_SOURCE 設為 'from_predict'
    if SAMPLE_SOURCE == 'from_predict':
        if not os.path.exists("acct_predict.csv"):
            raise RuntimeError("SAMPLE_SOURCE='from_predict' 時需有 acct_predict.csv 檔案在工作目錄。")
        ap = pd.read_csv("acct_predict.csv", dtype=str).fillna("")
        pred_list = ap['acct'].astype(str).tolist()
        # 只從 predict 清單中取，並排除正例
        neg_candidates = [a for a in pred_list if a not in pos_set]
        print(f"[info] 使用 acct_predict 作為負例候選，排除正例後候選數: {len(neg_candidates)}")
    if len(neg_candidates) < neg_n:
        print(f"[warn] 候選數 {len(neg_candidates)} 少於目標 neg_n {neg_n}，將使用全部候選 ({len(neg_candidates)})。")
        neg_n = len(neg_candidates)
    neg_sampled = random.sample(neg_candidates, neg_n)
    print(f"[info] 隨機抽樣負例完成，共 {len(neg_sampled)} 筆")
elif SAMPLE_SOURCE == 'match_activity':
    # 計算每個帳戶的交易次數（逐 chunk）
    print("[info] 使用 match_activity：先計算每帳戶在母集的交易次數分布（需要掃描母集一次）...")
    cnt = Counter()
    for chunk in pd.read_csv(MASTER_TXN_CSV, dtype=str, chunksize=CHUNK_SIZE):
        chunk = chunk.fillna("")
        cnt.update(chunk['from_acct'].astype(str).tolist())
        cnt.update(chunk['to_acct'].astype(str).tolist())
    # 取出負例候選的計數
    cand_counts = [(a, cnt.get(a,0)) for a in neg_candidates]
    # 做 quantile bins
    import numpy as np
    counts_arr = np.array([c for _,c in cand_counts])
    if len(counts_arr) == 0:
        raise RuntimeError("候選帳戶計數陣列為空。")
    bins = np.quantile(counts_arr, [0,0.2,0.4,0.6,0.8,1.0])
    binned = {}
    for a,c in cand_counts:
        idx = int(np.searchsorted(bins, c, side='right') - 1)
        binned.setdefault(idx, []).append(a)
    # 正例 bin 分布
    pos_counts = [cnt.get(a,0) for a in pos_accts]
    pos_bins = np.searchsorted(bins, pos_counts, side='right') - 1
    from collections import Counter as _Counter
    pos_bin_counts = _Counter(pos_bins)
    neg_sampled = []
    for bin_idx, pcnt in pos_bin_counts.items():
        want = int(round(pcnt / max(1,len(pos_accts)) * neg_n))
        pool = binned.get(bin_idx, [])
        if len(pool) == 0:
            continue
        if want >= len(pool):
            neg_sampled.extend(pool)
        else:
            neg_sampled.extend(random.sample(pool, want))
    # 補足填滿數量
    neg_sampled = list(dict.fromkeys(neg_sampled))
    if len(neg_sampled) < neg_n:
        remaining = [a for a in neg_candidates if a not in set(neg_sampled)]
        need = neg_n - len(neg_sampled)
        if len(remaining) <= need:
            neg_sampled.extend(remaining)
        else:
            neg_sampled.extend(random.sample(remaining, need))
    print(f"[info] match_activity 抽樣完成，共 {len(neg_sampled)} 筆")
else:
    raise ValueError("SAMPLE_SOURCE 必須是 'random' | 'match_activity' | 'from_predict'。")

# 5) 輸出 acct_normal.csv
acct_normal_df = pd.DataFrame({'acct': neg_sampled, 'label': 0})
acct_normal_df.to_csv(OUT_ACCT_NORMAL, index=False)
print(f"[done] 已輸出負例帳戶檔案: {OUT_ACCT_NORMAL} (共 {len(acct_normal_df)} 筆)")

# 6) 從母集逐 chunk 篩選負例帳戶的交易並寫入 normal_transaction.csv
print("[info] 正在從母集篩選負例帳戶的交易（chunked 寫入）… 這會再次掃描母集檔案。")
sample_set = set(neg_sampled)
first_write = True
rows_written = 0
for chunk in pd.read_csv(MASTER_TXN_CSV, dtype=str, chunksize=CHUNK_SIZE):
    chunk = chunk.fillna("")
    mask = chunk['from_acct'].isin(sample_set) | chunk['to_acct'].isin(sample_set)
    sel = chunk.loc[mask]
    if not sel.empty:
        sel.to_csv(OUT_NORMAL_TXN, mode='w' if first_write else 'a', header=first_write, index=False)
        first_write = False
        rows_written += len(sel)
print(f"[done] 篩選完成，已寫入 {rows_written} 筆交易到 {OUT_NORMAL_TXN}")


[info] 使用 kernel 中的 acct_alert，共 1004 筆正例。
[info] 目標負例數 neg_n = 1004
[info] 開始掃描母集以蒐集候選帳戶（此動作會逐 chunk 讀取檔案，視檔案大小可能需要數十秒至數分鐘）...
[info] 母集帳戶總數（包含正例）: 1800106
[info] 去除正例後的負例候選數: 1799102
[info] 隨機抽樣負例完成，共 1004 筆
[done] 已輸出負例帳戶檔案: acct_normal.csv (共 1004 筆)
[info] 正在從母集篩選負例帳戶的交易（chunked 寫入）… 這會再次掃描母集檔案。
[done] 篩選完成，已寫入 4367 筆交易到 normal_transaction.csv


In [21]:
# Debug + 修正 cell：檢查 acct_alert 的欄位名稱，標準化欄名，解析日期，並修正 append -> concat 問題
import pandas as pd
import numpy as np
import random
from collections import defaultdict

# 調整以下路徑（若已在 kernel 有變數 acct_alert，會優先使用）
ACCT_ALERT_CSV = "acct_alert.csv"
ALERT_TXN_CSV = "alert_transaction.csv"
NORMAL_TXN_CSV = "normal_transaction.csv"

# 讀 acct_alert（若 kernel 中已存在 acct_alert DataFrame，印出其欄位與 head）
if 'acct_alert' in globals() and isinstance(acct_alert, pd.DataFrame):
    print("[info] kernel 中已有 acct_alert 變數，使用它（不重新從 CSV 讀取）")
    acct_alert_df = acct_alert.copy()
else:
    print(f"[info] 從檔案讀取 {ACCT_ALERT_CSV}")
    acct_alert_df = pd.read_csv(ACCT_ALERT_CSV, dtype=str, keep_default_na=False).fillna("")

# 列出原始欄位和前 5 列供檢查
print("原始 acct_alert 欄位：", list(acct_alert_df.columns))
print("acct_alert 前 5 列：")
display(acct_alert_df.head())

# Normalize 欄位名稱（strip, lower, replace spaces -> underscore）
def normalize_col(c):
    if not isinstance(c, str):
        c = str(c)
    c2 = c.strip().lower().replace(" ", "_").replace("-", "_")
    # optionally remove BOM or weird chars
    c2 = c2.replace("\ufeff", "")
    return c2

orig_cols = list(acct_alert_df.columns)
norm_cols = [normalize_col(c) for c in orig_cols]
mapping = dict(zip(orig_cols, norm_cols))
acct_alert_df = acct_alert_df.rename(columns=mapping)
print("標準化後欄位：", list(acct_alert_df.columns))

# 嘗試自動把最可能的欄位改成 event_date（若已存在 'event_date' 則保留）
if 'event_date' not in acct_alert_df.columns:
    # 找出包含 'event' 的欄位
    candidates = [c for c in acct_alert_df.columns if 'event' in c or ('alert' in c and 'date' in c) or ('eventdate' in c)]
    if candidates:
        chosen = candidates[0]
        acct_alert_df = acct_alert_df.rename(columns={chosen: 'event_date'})
        print(f"[fix] 將欄位 '{chosen}' 重新命名為 'event_date'")
    else:
        # 也試找含 'date' 或 'time' 的欄位（但需提醒使用者）
        candidates2 = [c for c in acct_alert_df.columns if 'date' in c or 'time' in c]
        if candidates2:
            print("[warn] 找不到包含 'event' 的欄位，但發現可能的日期欄位: ", candidates2)
            print("請確認哪個欄位是 event_date，或手動 rename，例如: acct_alert_df.rename(columns={'你的欄位':'event_date'}, inplace=True)")
        else:
            print("[warn] 找不到任何可能的日期欄位 (event_date)。請檢查 acct_alert.csv 的 header 是否如預期。")

# 顯示最終欄位與前 5 列
print("最終 acct_alert 欄位：", list(acct_alert_df.columns))
display(acct_alert_df.head())

# 解析 event_date（如果有）
if 'event_date' in acct_alert_df.columns:
    # 嘗試解析，使用 dayfirst=False 並允許 dateutil fallback；若格式一致可改加 format 參數
    acct_alert_df['event_date_parsed'] = pd.to_datetime(acct_alert_df['event_date'], errors='coerce', infer_datetime_format=True)
    n_missing = acct_alert_df['event_date_parsed'].isna().sum()
    print(f"event_date 欄解析完成：成功 {len(acct_alert_df)-n_missing}, 失敗 {n_missing}")
    # 顯示前幾個解析失敗的原始值以供檢查
    if n_missing > 0:
        bad = acct_alert_df.loc[acct_alert_df['event_date_parsed'].isna(), 'event_date'].unique()[:10]
        print("解析失敗的 event_date 範例（最多顯示 10 個）:", bad)
else:
    print("acct_alert 中仍然沒有 event_date 欄，請手動檢查或貼上檔頭讓我幫你判斷。")

# ------------- 交易檔的時間欄解析 -------------
def ensure_dt_col(df, candidate_names=['dt','txn_date','txn_datetime','date','trade_date','tran_date','time']):
    # 找出一個存在的欄位並轉成 dt
    for c in candidate_names:
        if c in df.columns:
            try:
                df['dt'] = pd.to_datetime(df[c], errors='coerce', infer_datetime_format=True)
                print(f"[info] 使用欄位 '{c}' 解析為 dt (成功解析非 NaT 的數量: {df['dt'].notna().sum()})")
                return df
            except Exception as e:
                print(f"[warn] 嘗試解析欄位 {c} 失敗: {e}")
    # 若都沒有，建立空的 dt
    df['dt'] = pd.NaT
    print("[warn] 找不到常見的時間欄位，已新增 dt=NaT")
    return df

print("\n解析 alert_transaction 的時間欄位...")
tx_alert = pd.read_csv(ALERT_TXN_CSV, dtype=str, keep_default_na=False).fillna("")
tx_alert = ensure_dt_col(tx_alert)

print("\n解析 normal_transaction 的時間欄位...")
tx_normal = pd.read_csv(NORMAL_TXN_CSV, dtype=str, keep_default_na=False).fillna("")
tx_normal = ensure_dt_col(tx_normal)

# 顯示 dt 欄解析狀況
print("alert tx dt 非 NaT 筆數:", tx_alert['dt'].notna().sum(), " / ", len(tx_alert))
print("normal tx dt 非 NaT 筆數:", tx_normal['dt'].notna().sum(), " / ", len(tx_normal))

# 若 dt 有大量 NaT，請貼出幾筆樣例的 txn_date 值讓我看格式
if tx_alert['dt'].notna().sum() < len(tx_alert):
    sample_bad = tx_alert.loc[tx_alert['dt'].isna(), :].head(5)
    print("alert tx 部分 dt 解析為 NaT，以下為前 5 筆原始時間欄位值（請貼回以便我判斷格式）:")
    display(sample_bad[[c for c in tx_alert.columns if 'date' in c or 'time' in c][:3]])

# ------------- 修正 append 為 concat 的範例 -------------
# 用於計算負例帳戶最後交易時間（fallback cutoff），之前用 append 導致錯誤
print("\n建立每個負例帳戶的最後交易時間（示範）...")
# 我們將 tx_normal 兩端合併以取得 per-account last dt
tmp_from = tx_normal[['from_acct','dt']].rename(columns={'from_acct':'acct'})
tmp_to   = tx_normal[['to_acct','dt']].rename(columns={'to_acct':'acct'})
tmp_all = pd.concat([tmp_from, tmp_to], ignore_index=True)
# 若 dt 有 NaT，這些會被忽略在 max 計算中
grp_last = tmp_all.groupby('acct')['dt'].max().to_dict()
print("已建立 grp_last（acct -> last dt），範例 5 筆：")
cnt = 0
for k,v in list(grp_last.items())[:5]:
    print(k, "->", v)
    cnt += 1
    if cnt>=5: break

# ------------- 最後診斷輸出，並提示下一步 -------------
print("\n診斷完畢總結：")
print("- acct_alert 是否包含 event_date 欄位（標準化後）:", 'event_date' in acct_alert_df.columns)
if 'event_date' in acct_alert_df.columns:
    print("  -> event_date parsed NaT count:", acct_alert_df['event_date_parsed'].isna().sum())
print("- alert_transaction dt 非 NaT 筆數:", tx_alert['dt'].notna().sum(), "/", len(tx_alert))
print("- normal_transaction dt 非 NaT 筆數:", tx_normal['dt'].notna().sum(), "/", len(tx_normal))

print("\n下一步建議：")
print("1) 若 acct_alert 中的 event_date 不是標準格式（或解析失敗），請貼出 acct_alert.head() 與幾個 event_date 值樣例，我會幫你指定 format 去解析。")
print("2) 若 tx_alert / tx_normal 的時間欄位解析有 NaT，請貼出幾筆原始時間字串範例（我會協助寫 to_datetime(format=...)）。")
print("3) 若你同意，我會把後續用 cutoff 計算特徵的 cell 以已修正的版本回傳給你（已修正 append->concat 與 event_date 解析）。")

[info] kernel 中已有 acct_alert 變數，使用它（不重新從 CSV 讀取）
原始 acct_alert 欄位： ['acct', 'event_date']
acct_alert 前 5 列：


,acct,event_date
0,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,87
1,b8c11db05d00b5ac66be10ffee5f6ce6ef9221c733a4bb...,19
2,daa05c68b290ac3cc522abad400c5304dffba07baa232c...,81
3,174e26ecc9cee56aaaca855c743a106275c58629740a49...,88
4,007cf5c98aa4f9f3e444c9cdaca74d0f7542e9a2804201...,117


標準化後欄位： ['acct', 'event_date']
最終 acct_alert 欄位： ['acct', 'event_date']


,acct,event_date
0,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,87
1,b8c11db05d00b5ac66be10ffee5f6ce6ef9221c733a4bb...,19
2,daa05c68b290ac3cc522abad400c5304dffba07baa232c...,81
3,174e26ecc9cee56aaaca855c743a106275c58629740a49...,88
4,007cf5c98aa4f9f3e444c9cdaca74d0f7542e9a2804201...,117


event_date 欄解析完成：成功 0, 失敗 1004
解析失敗的 event_date 範例（最多顯示 10 個）: ['87' '19' '81' '88' '117' '100' '65' '85' '32' '110']

解析 alert_transaction 的時間欄位...
[info] 使用欄位 'txn_date' 解析為 dt (成功解析非 NaT 的數量: 0)

解析 normal_transaction 的時間欄位...
[info] 使用欄位 'txn_date' 解析為 dt (成功解析非 NaT 的數量: 0)
alert tx dt 非 NaT 筆數: 0  /  33933
normal tx dt 非 NaT 筆數: 0  /  4367
alert tx 部分 dt 解析為 NaT，以下為前 5 筆原始時間欄位值（請貼回以便我判斷格式）:


C:\Users\macrj\AppData\Local\Temp\ipykernel_35428\1515409020.py:64: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  acct_alert_df['event_date_parsed'] = pd.to_datetime(acct_alert_df['event_date'], errors='coerce', infer_datetime_format=True)
C:\Users\macrj\AppData\Local\Temp\ipykernel_35428\1515409020.py:64: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  acct_alert_df['event_date_parsed'] = pd.to_datetime(acct_alert_df['event_date'], errors='coerce', infer_datetime_format=True)
C:\Users\macrj\AppData\Local\Temp\ipykernel_35428\1515409020.py:80: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a futu

,txn_date,txn_time
0,84,13:05:00
1,85,10:55:00
2,86,13:05:00
3,86,13:30:00
4,86,13:10:00



建立每個負例帳戶的最後交易時間（示範）...
已建立 grp_last（acct -> last dt），範例 5 筆：
0001caf55cf6007357c94bd591e3a1d057d3cb2e54309a80e4097827158989ac -> NaT
00036456b78190165c6f8332d000b6c4659f5102093b19c2248b470a6a322f35 -> NaT
0007b1ca10bfc61ea4c83c7e5a4dc42b97b274e904e5757b5738a5ec5b98dc4e -> NaT
001b33483eb2fb642a0dfc247e062b72602a82793fe4657099264bd9ab900247 -> NaT
003ae68ab59886311bb85ed1e77872e4084ec912004c1108acd3a991667d2e28 -> NaT

診斷完畢總結：
- acct_alert 是否包含 event_date 欄位（標準化後）: True
  -> event_date parsed NaT count: 1004
- alert_transaction dt 非 NaT 筆數: 0 / 33933
- normal_transaction dt 非 NaT 筆數: 0 / 4367

下一步建議：
1) 若 acct_alert 中的 event_date 不是標準格式（或解析失敗），請貼出 acct_alert.head() 與幾個 event_date 值樣例，我會幫你指定 format 去解析。
2) 若 tx_alert / tx_normal 的時間欄位解析有 NaT，請貼出幾筆原始時間字串範例（我會協助寫 to_datetime(format=...)）。
3) 若你同意，我會把後續用 cutoff 計算特徵的 cell 以已修正的版本回傳給你（已修正 append->concat 與 event_date 解析）。


In [22]:
# Cell: 自動偵測並嘗試解析 event_date 與交易時間欄位（貼到 notebook 執行）
import pandas as pd
import numpy as np
import math
from datetime import datetime
import re

ACCT_ALERT_CSV = "acct_alert.csv"
ALERT_TXN_CSV = "alert_transaction.csv"
NORMAL_TXN_CSV = "normal_transaction.csv"

# 讀檔（若 kernel 有變數則用它）
if 'acct_alert' in globals() and isinstance(acct_alert, pd.DataFrame):
    acct_alert_df = acct_alert.copy()
else:
    acct_alert_df = pd.read_csv(ACCT_ALERT_CSV, dtype=str, keep_default_na=False).fillna("")

tx_alert = pd.read_csv(ALERT_TXN_CSV, dtype=str, keep_default_na=False).fillna("")
tx_normal = pd.read_csv(NORMAL_TXN_CSV, dtype=str, keep_default_na=False).fillna("")

print("acct_alert cols:", list(acct_alert_df.columns))
print("alert tx cols:", list(tx_alert.columns)[:50])
print("normal tx cols:", list(tx_normal.columns)[:50])

# Normalize col names like before
def norm_cols(df):
    return [ (c.strip().lower().replace(" ", "_").replace("-", "_").replace("\ufeff","")) for c in df.columns ]
acct_alert_df.columns = norm_cols(acct_alert_df)
tx_alert.columns = norm_cols(tx_alert)
tx_normal.columns = norm_cols(tx_normal)

print("\n標準化後 acct_alert cols:", list(acct_alert_df.columns))
print("標準化後 alert tx cols:", list(tx_alert.columns)[:50])

# Show sample values from acct_alert's event_date-like columns
possible_event_cols = [c for c in acct_alert_df.columns if 'event' in c or ('alert' in c and 'date' in c) or c in ('event_date','eventdate','date','dt')]
print("\nacct_alert 可能的時間欄位:", possible_event_cols)
for c in possible_event_cols:
    print(f"\n--- 顯示 acct_alert['{c}'] 前 20 個原始值 ---")
    display(acct_alert_df[c].head(20))

# Show candidate txn datetime columns in transactions
def find_time_cols(df):
    cands = [c for c in df.columns if any(k in c for k in ('dt','date','time','datetime','txn'))]
    # return unique
    return cands

tx_time_cols = sorted(set(find_time_cols(tx_alert) + find_time_cols(tx_normal)))
print("\n交易檔可能的時間欄位:", tx_time_cols)
for c in tx_time_cols:
    print(f"\n--- alert tx sample for '{c}' ---")
    display(tx_alert[c].head(10))
    print(f"--- normal tx sample for '{c}' ---")
    display(tx_normal[c].head(10))

# Helper: try parsing a series with multiple formats and strategies, report success counts
from dateutil import parser

def try_parsers(series, name="series"):
    s = series.astype(str).replace("", np.nan)
    s_nonnull = s.dropna()
    total = len(s)
    print(f"\n解析嘗試 for {name}: 總筆數 {total}, 非空 {len(s_nonnull)}")
    results = {}
    # 1) pandas general infer
    parsed = pd.to_datetime(s, errors='coerce', infer_datetime_format=True)
    results['pd_infer'] = parsed.notna().sum()
    # 2) try common formats list
    common = [
        "%Y-%m-%d %H:%M:%S",
        "%Y/%m/%d %H:%M:%S",
        "%Y-%m-%d",
        "%Y/%m/%d",
        "%Y%m%d",
        "%Y%m%d%H%M%S",
        "%d/%m/%Y",
        "%m/%d/%Y",
        "%d-%b-%Y %H:%M:%S",
        "%Y-%m-%dT%H:%M:%S%z",
        "%Y-%m-%dT%H:%M:%S",  # ISO without tz
        "%Y.%m.%d %H:%M:%S",
    ]
    for fmt in common:
        try:
            p = pd.to_datetime(s, format=fmt, errors='coerce')
            results[f'fmt:{fmt}'] = p.notna().sum()
        except Exception:
            results[f'fmt:{fmt}'] = 0
    # 3) numeric timestamps: detect if values are integers (seconds or ms)
    # sample non-null values
    sample_vals = s_nonnull.head(200).tolist()
    numeric_count = 0
    as_epoch_sec = 0
    as_epoch_ms = 0
    for v in sample_vals:
        v0 = re.sub(r'[^\d\-]', '', str(v))  # keep digits and minus
        if v0=="":
            continue
        if re.fullmatch(r'-?\d+', v0):
            numeric_count += 1
            # inspect length
            if len(v0) >= 13:  # ms
                try:
                    t = pd.to_datetime(int(v0), unit='ms')
                    if not pd.isna(t):
                        as_epoch_ms += 1
                except:
                    pass
            if len(v0) >= 10 and len(v0) <= 12:  # sec or ms
                try:
                    t2 = pd.to_datetime(int(v0), unit='s')
                    if not pd.isna(t2):
                        as_epoch_sec += 1
                except:
                    pass
    results['sample_numeric_values'] = numeric_count
    results['sample_epoch_sec_success'] = as_epoch_sec
    results['sample_epoch_ms_success'] = as_epoch_ms
    # 4) fallback: dateutil parser attempt (slower) on sample
    sample_parsed = 0
    for v in sample_vals:
        try:
            parser.parse(v)
            sample_parsed += 1
        except:
            pass
    results['dateutil_sample_success'] = sample_parsed
    # print results sorted
    for k,v in sorted(results.items(), key=lambda x: (-x[1] if isinstance(x[1], int) else 0, x[0])):
        print(f"{k}: {v}")
    return results

# Try parsers on acct_alert.event_date candidate (if exists)
if 'event_date' in acct_alert_df.columns:
    print("\n=== 嘗試解析 acct_alert['event_date'] ===")
    try_parsers(acct_alert_df['event_date'])
else:
    print("\nacct_alert 沒有 event_date 欄可解析。")

# Try parsers on transaction candidate columns (show summary per column)
for c in tx_time_cols:
    print(f"\n=== 嘗試解析 tx column '{c}'（alert sample）=== ")
    try_parsers(tx_alert[c], name=f"alert.{c}")
    print(f"=== 嘗試解析 tx column '{c}'（normal sample）=== ")
    try_parsers(tx_normal[c], name=f"normal.{c}")

print("\n完成解析嘗試。請檢查上面每個格式的成功筆數 (非 NaT)。")
print("回傳最有成功筆數（非 NaT） 的格式名稱或貼上 acct_alert.event_date 與交易檔中代表性的原始時間字串（例如前 10 個），我會給你精準的 pd.to_datetime(..., format='...')。")


acct_alert cols: ['acct', 'event_date']
alert tx cols: ['from_acct', 'from_acct_type', 'to_acct', 'to_acct_type', 'is_self_txn', 'txn_amt', 'txn_date', 'txn_time', 'currency_type', 'channel_type']
normal tx cols: ['from_acct', 'from_acct_type', 'to_acct', 'to_acct_type', 'is_self_txn', 'txn_amt', 'txn_date', 'txn_time', 'currency_type', 'channel_type']

標準化後 acct_alert cols: ['acct', 'event_date']
標準化後 alert tx cols: ['from_acct', 'from_acct_type', 'to_acct', 'to_acct_type', 'is_self_txn', 'txn_amt', 'txn_date', 'txn_time', 'currency_type', 'channel_type']

acct_alert 可能的時間欄位: ['event_date']

--- 顯示 acct_alert['event_date'] 前 20 個原始值 ---


0      87
1      19
2      81
3      88
4     117
5     100
6      65
7      85
8      32
9     110
10     99
11     62
12     88
13     67
14     81
15    115
16     80
17    108
18     71
19    101
Name: event_date, dtype: object


交易檔可能的時間欄位: ['is_self_txn', 'txn_amt', 'txn_date', 'txn_time']

--- alert tx sample for 'is_self_txn' ---


0    UNK
1    UNK
2    UNK
3    UNK
4    UNK
5    UNK
6    UNK
7    UNK
8    UNK
9    UNK
Name: is_self_txn, dtype: object

--- normal tx sample for 'is_self_txn' ---


...

0    16:15:00
1    15:15:00
2    13:30:00
3    12:25:00
4    19:25:00
5    13:35:00
6    16:10:00
7    15:35:00
8    11:30:00
9    16:20:00
Name: txn_time, dtype: object


=== 嘗試解析 acct_alert['event_date'] ===

解析嘗試 for series: 總筆數 1004, 非空 1004
dateutil_sample_success: 200
sample_numeric_values: 200
fmt:%Y%m%d: 0
fmt:%Y%m%d%H%M%S: 0
fmt:%Y-%m-%d: 0
fmt:%Y-%m-%d %H:%M:%S: 0
fmt:%Y-%m-%dT%H:%M:%S: 0
fmt:%Y-%m-%dT%H:%M:%S%z: 0
fmt:%Y.%m.%d %H:%M:%S: 0
fmt:%Y/%m/%d: 0
fmt:%Y/%m/%d %H:%M:%S: 0
fmt:%d-%b-%Y %H:%M:%S: 0
fmt:%d/%m/%Y: 0
fmt:%m/%d/%Y: 0
pd_infer: 0
sample_epoch_ms_success: 0
sample_epoch_sec_success: 0

=== 嘗試解析 tx column 'is_self_txn'（alert sample）=== 

解析嘗試 for alert.is_self_txn: 總筆數 33933, 非空 33933
dateutil_sample_success: 0
fmt:%Y%m%d: 0
fmt:%Y%m%d%H%M%S: 0
fmt:%Y-%m-%d: 0
fmt:%Y-%m-%d %H:%M:%S: 0
fmt:%Y-%m-%dT%H:%M:%S: 0
fmt:%Y-%m-%dT%H:%M:%S%z: 0
fmt:%Y.%m.%d %H:%M:%S: 0
fmt:%Y/%m/%d: 0
fmt:%Y/%m/%d %H:%M:%S: 0
fmt:%d-%b-%Y %H:%M:%S: 0
fmt:%d/%m/%Y: 0
fmt:%m/%d/%Y: 0
pd_infer: 0
sample_epoch_ms_success: 0
sample_epoch_sec_success: 0
sample_numeric_values: 0
=== 嘗試解析 tx column 'is_self_txn'（normal sample）=== 

解析嘗試 for normal.is_self_txn: 

C:\Users\macrj\AppData\Local\Temp\ipykernel_35428\1986107349.py:66: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(s, errors='coerce', infer_datetime_format=True)
C:\Users\macrj\AppData\Local\Temp\ipykernel_35428\1986107349.py:66: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors='coerce', infer_datetime_format=True)
C:\Users\macrj\AppData\Local\Temp\ipykernel_35428\1986107349.py:66: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-t

dateutil_sample_success: 200
sample_numeric_values: 200
fmt:%Y%m%d: 0
fmt:%Y%m%d%H%M%S: 0
fmt:%Y-%m-%d: 0
fmt:%Y-%m-%d %H:%M:%S: 0
fmt:%Y-%m-%dT%H:%M:%S: 0
fmt:%Y-%m-%dT%H:%M:%S%z: 0
fmt:%Y.%m.%d %H:%M:%S: 0
fmt:%Y/%m/%d: 0
fmt:%Y/%m/%d %H:%M:%S: 0
fmt:%d-%b-%Y %H:%M:%S: 0
fmt:%d/%m/%Y: 0
fmt:%m/%d/%Y: 0
pd_infer: 33933
sample_epoch_ms_success: 0
sample_epoch_sec_success: 0
=== 嘗試解析 tx column 'txn_time'（normal sample）=== 

解析嘗試 for normal.txn_time: 總筆數 4367, 非空 4367
dateutil_sample_success: 200
sample_numeric_values: 200
fmt:%Y%m%d: 0
fmt:%Y%m%d%H%M%S: 0
fmt:%Y-%m-%d: 0
fmt:%Y-%m-%d %H:%M:%S: 0
fmt:%Y-%m-%dT%H:%M:%S: 0
fmt:%Y-%m-%dT%H:%M:%S%z: 0
fmt:%Y.%m.%d %H:%M:%S: 0
fmt:%Y/%m/%d: 0
fmt:%Y/%m/%d %H:%M:%S: 0
fmt:%d-%b-%Y %H:%M:%S: 0
fmt:%d/%m/%Y: 0
fmt:%m/%d/%Y: 0
pd_infer: 4367
sample_epoch_ms_success: 0
sample_epoch_sec_success: 0

完成解析嘗試。請檢查上面每個格式的成功筆數 (非 NaT)。
回傳最有成功筆數（非 NaT） 的格式名稱或貼上 acct_alert.event_date 與交易檔中代表性的原始時間字串（例如前 10 個），我會給你精準的 pd.to_datetime(..., format='...')。


C:\Users\macrj\AppData\Local\Temp\ipykernel_35428\1986107349.py:66: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  parsed = pd.to_datetime(s, errors='coerce', infer_datetime_format=True)
C:\Users\macrj\AppData\Local\Temp\ipykernel_35428\1986107349.py:66: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(s, errors='coerce', infer_datetime_format=True)


In [23]:
# Cell: 解析 event_date 與 txn_date+txn_time 為 datetime（把 offset 轉回實際日期）
import pandas as pd
import numpy as np

# 若你在 kernel 有 BASE_DATE，會直接使用；否則使用下面預設（請依你的情況調整）
BASE_DATE = globals().get('BASE_DATE', '2020-01-01')
base = pd.to_datetime(BASE_DATE)

# 讀取檔案（若已在 kernel 有變數 acct_alert / tx_alert / tx_normal 則優先使用它們）
if 'acct_alert' in globals() and isinstance(acct_alert, pd.DataFrame):
    acct_alert_df = acct_alert.copy()
else:
    acct_alert_df = pd.read_csv("acct_alert.csv", dtype=str).fillna("")

tx_alert = pd.read_csv("alert_transaction.csv", dtype=str).fillna("")
tx_normal = pd.read_csv("normal_transaction.csv", dtype=str).fillna("")

# 1) 解析 acct_alert.event_date：先轉成整數 offset（若原檔即為 offset）
acct_alert_df['event_date_int'] = pd.to_numeric(acct_alert_df['event_date'], errors='coerce')
# 若 event_date 原本就是像 87, 19 的「第 N 天偏移」，把它轉為實際日期（base + (n-1) days）
acct_alert_df['event_date_dt'] = acct_alert_df['event_date_int'].apply(
    lambda x: (base + pd.Timedelta(days=int(x)-1)) if not pd.isna(x) else pd.NaT
)

print("acct_alert: total rows", len(acct_alert_df),
      ", parsed event_date_int non-null:", acct_alert_df['event_date_int'].notna().sum())
print("acct_alert event_date_dt sample:")
display(acct_alert_df[['acct','event_date','event_date_int','event_date_dt']].head(8))

# 2) 解析交易檔的 txn_date 與 txn_time -> 建立 dt
def build_dt_from_date_time(df):
    # 把 txn_date 轉成整數（offset），txn_time 若空值補 00:00:00
    df['txn_date_int'] = pd.to_numeric(df.get('txn_date',''), errors='coerce')
    df['txn_time'] = df.get('txn_time','').replace('', '00:00:00')
    # 安全轉 timedelta（若格式正確如 '13:05:00'，pd.to_timedelta 可解析）
    def safe_time_to_timedelta(t):
        try:
            return pd.to_timedelta(t)
        except Exception:
            return pd.to_timedelta('00:00:00')
    df['time_only'] = df['txn_time'].apply(safe_time_to_timedelta)
    # 若 txn_date 為 offset 整數，建立 dt = base + (txn_date_int - 1) days + time_only
    df['dt'] = df['txn_date_int'].apply(
        lambda x: (base + pd.Timedelta(days=int(x)-1)) if not pd.isna(x) else pd.NaT
    ) + df['time_only']
    return df

tx_alert = build_dt_from_date_time(tx_alert)
tx_normal = build_dt_from_date_time(tx_normal)

print("alert tx rows:", len(tx_alert), "parsed dt non-null:", tx_alert['dt'].notna().sum())
print("normal tx rows:", len(tx_normal), "parsed dt non-null:", tx_normal['dt'].notna().sum())
print("alert txn sample (first 8 dt / txn_date / txn_time):")
display(tx_alert[['from_acct','to_acct','txn_date','txn_time','txn_date_int','dt']].head(8))
print("normal txn sample (first 8 dt / txn_date / txn_time):")
display(tx_normal[['from_acct','to_acct','txn_date','txn_time','txn_date_int','dt']].head(8))

# 若解析結果仍有大量 NaT，請把上述 display 的原始欄位值貼回給我，我會協助進一步處理。
# 儲存回 kernel 變數，方便下個 cell 使用
acct_alert = acct_alert_df
alert_tx = tx_alert
normal_tx = tx_normal

acct_alert: total rows 1004 , parsed event_date_int non-null: 1004
acct_alert event_date_dt sample:


,acct,event_date,event_date_int,event_date_dt
0,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,87,87,2020-03-27
1,b8c11db05d00b5ac66be10ffee5f6ce6ef9221c733a4bb...,19,19,2020-01-19
2,daa05c68b290ac3cc522abad400c5304dffba07baa232c...,81,81,2020-03-21
3,174e26ecc9cee56aaaca855c743a106275c58629740a49...,88,88,2020-03-28
4,007cf5c98aa4f9f3e444c9cdaca74d0f7542e9a2804201...,117,117,2020-04-26
5,3cc3d14ff57d2518c18b0352383979bac4c102ae890205...,100,100,2020-04-09
6,fd6da4c149c34652b3e0482bea500e5d65d2f8a05311dd...,65,65,2020-03-05
7,82f43d1cccc8b43a6b863a0220941a537afa96dd39e3fa...,85,85,2020-03-25


alert tx rows: 33933 parsed dt non-null: 33933
normal tx rows: 4367 parsed dt non-null: 4367
alert txn sample (first 8 dt / txn_date / txn_time):


,from_acct,to_acct,txn_date,txn_time,txn_date_int,dt
0,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,d66ae51046671ae1cfe50f04b2f3da6d28652328ca6966...,84,13:05:00,84,2020-03-24 13:05:00
1,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,d66ae51046671ae1cfe50f04b2f3da6d28652328ca6966...,85,10:55:00,85,2020-03-25 10:55:00
2,219e2289c8476ea9f3a946203ea2ce3544af58d6091978...,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,86,13:05:00,86,2020-03-26 13:05:00
3,79e65f376560e0a5becbd8b8c6130c34057201abb154fc...,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,86,13:30:00,86,2020-03-26 13:30:00
4,219e2289c8476ea9f3a946203ea2ce3544af58d6091978...,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,86,13:10:00,86,2020-03-26 13:10:00
5,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,118a694d9816ba6afdaa9ed137aabcac8683a5a0cf9dec...,86,11:30:00,86,2020-03-26 11:30:00
6,9a4980320f5501ca8f0d57f1ae52050828864f5c846cb6...,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,86,13:25:00,86,2020-03-26 13:25:00
7,9a4980320f5501ca8f0d57f1ae52050828864f5c846cb6...,80bd1c28b47357a3d37a01835ebb1bed5edf54e791bd3d...,86,13:35:00,86,2020-03-26 13:35:00


normal txn sample (first 8 dt / txn_date / txn_time):


,from_acct,to_acct,txn_date,txn_time,txn_date_int,dt
0,d6c80b3924158141d2c72058b6ef27a87563f4b32a0f9f...,609b4220b036dfd51dd0d11796696abc8c08b2abf0ec31...,99,16:15:00,99,2020-04-08 16:15:00
1,b5cdcd54d8e9c6e98ec7e8fe16335ce18db66d68f53df3...,6d6bef120a9bca411ecb9f0333ff22587b92801baca724...,66,15:15:00,66,2020-03-06 15:15:00
2,dfcf91e4638d7227fa1b42d7b5da2d801cb6803e217330...,2f14d2f87442479e1ee6ad94ac43c34815f304ef819bcf...,83,13:30:00,83,2020-03-23 13:30:00
3,e9bc0abf1adf44a4656bf5cb7d2424e709a7eda2b9af1c...,583fa07cb76858504738b9c2db7ce41851ab202c14e1ad...,74,12:25:00,74,2020-03-14 12:25:00
4,9e90b62575f03346895a74058dfcec3cb5b93f2eaf5f47...,447cfdd22e00de8c1e61c2a48c544f44f9dd7fb8a12965...,66,19:25:00,66,2020-03-06 19:25:00
5,f5a6fc9b18abf7e467c74fbb2402372b3bc608d5da26f3...,c4a1ebe2cb3ac670b6ffafb70f9bf70ed60c24bf3878c7...,72,13:35:00,72,2020-03-12 13:35:00
6,3984b459cb0f152db9bf22a2c82962c248b1153b22a999...,dbc2cb5d0608037536618d6fb61a16ae3d56dffd057262...,87,16:10:00,87,2020-03-27 16:10:00
7,392d274c93b1d7fdb8a3806152a5fd6a20b7348923dd5c...,9be775ccd9b84932763dd0d03c7bbbf98d89ddfe8a799c...,87,15:35:00,87,2020-03-27 15:35:00


In [3]:
# 一個可直接執行的「檢查/建立變數 + cutoff-aware 特徵化 + baseline」cell
# 請把整段貼到 notebook 新 cell 並執行
import os
import pandas as pd
import numpy as np
import random
import joblib
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report

# safe tqdm import
try:
    from tqdm.auto import tqdm
except Exception:
    try:
        from tqdm import tqdm
    except Exception:
        def tqdm(iterable, *args, **kwargs): return iterable

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# File names (如檔案不在工作目錄請改路徑)
ACCT_ALERT_CSV = "acct_alert.csv"
ALERT_TXN_CSV = "alert_transaction.csv"
NORMAL_TXN_CSV = "normal_transaction.csv"
ACCT_NORMAL_CSV = "acct_normal.csv"
# 若 kernel 中已有 BASE_DATE，可直接使用；否則預設
BASE_DATE = globals().get('BASE_DATE', '2020-01-01')
base = pd.to_datetime(BASE_DATE)

# ------------- 1) 讀入或使用現有變數 -------------
# acct_alert
if 'acct_alert' in globals() and isinstance(acct_alert, pd.DataFrame):
    acct_alert_df = acct_alert.copy()
    print("[info] 使用 kernel 中的 acct_alert 變數")
else:
    if os.path.exists(ACCT_ALERT_CSV):
        acct_alert_df = pd.read_csv(ACCT_ALERT_CSV, dtype=str).fillna("")
        print(f"[info] 從檔案讀取 {ACCT_ALERT_CSV}")
    else:
        raise FileNotFoundError(f"找不到 {ACCT_ALERT_CSV}，請放入工作目錄或先建立 acct_alert 變數。")

# tx files
if 'alert_tx' in globals() and isinstance(alert_tx, pd.DataFrame):
    tx_alert = alert_tx.copy()
    print("[info] 使用 kernel 中的 alert_tx 變數")
else:
    if os.path.exists(ALERT_TXN_CSV):
        tx_alert = pd.read_csv(ALERT_TXN_CSV, dtype=str).fillna("")
        print(f"[info] 從檔案讀取 {ALERT_TXN_CSV}")
    else:
        raise FileNotFoundError(f"找不到 {ALERT_TXN_CSV}")

if 'normal_tx' in globals() and isinstance(normal_tx, pd.DataFrame):
    tx_normal = normal_tx.copy()
    print("[info] 使用 kernel 中的 normal_tx 變數")
else:
    if os.path.exists(NORMAL_TXN_CSV):
        tx_normal = pd.read_csv(NORMAL_TXN_CSV, dtype=str).fillna("")
        print(f"[info] 從檔案讀取 {NORMAL_TXN_CSV}")
    else:
        raise FileNotFoundError(f"找不到 {NORMAL_TXN_CSV}")

# acct_normal list
if os.path.exists(ACCT_NORMAL_CSV):
    acct_normal_df = pd.read_csv(ACCT_NORMAL_CSV, dtype=str).fillna("")
    print(f"[info] 從檔案讀取 {ACCT_NORMAL_CSV}")
else:
    if 'acct_normal' in globals() and isinstance(acct_normal, pd.DataFrame):
        acct_normal_df = acct_normal.copy()
        print("[info] 使用 kernel 中的 acct_normal 變數")
    else:
        raise FileNotFoundError(f"找不到 {ACCT_NORMAL_CSV}")

# ------------- 2) 標準化欄位名稱（小寫、去空白） -------------
def norm_cols(df):
    df = df.copy()
    newcols = {}
    for c in df.columns:
        c2 = str(c).strip().lower().replace(" ", "_").replace("-", "_").replace("\ufeff","")
        newcols[c] = c2
    df.rename(columns=newcols, inplace=True)
    return df

acct_alert_df = norm_cols(acct_alert_df)
tx_alert = norm_cols(tx_alert)
tx_normal = norm_cols(tx_normal)
acct_normal_df = norm_cols(acct_normal_df)

print("acct_alert 欄位：", list(acct_alert_df.columns))
print("alert_tx 欄位（前20）：", list(tx_alert.columns)[:20])
print("normal_tx 欄位（前20）：", list(tx_normal.columns)[:20])

# ------------- 3) 解析 event_date（offset 整數 → 實際日期） -------------
# 嘗試把 event_date 轉為整數 offset，然後加 BASE_DATE
if 'event_date' not in acct_alert_df.columns:
    # 若欄位名稱不同，嘗試猜測
    for c in acct_alert_df.columns:
        if 'event' in c and 'date' in c:
            acct_alert_df.rename(columns={c:'event_date'}, inplace=True)
            print(f"[fix] 將欄位 {c} 重新命名為 event_date")
            break

acct_alert_df['event_date_int'] = pd.to_numeric(acct_alert_df.get('event_date',''), errors='coerce')
acct_alert_df['event_date_dt'] = acct_alert_df['event_date_int'].apply(
    lambda x: (base + pd.Timedelta(days=int(x)-1)) if not pd.isna(x) else pd.NaT
)
print("parsed event_date_dt 非 NaT:", acct_alert_df['event_date_dt'].notna().sum(), "/", len(acct_alert_df))

# ------------- 4) 解析 txn_date + txn_time -> dt（若 txn_date 為 offset）-------------
def build_dt_from_date_time(df):
    df = df.copy()
    # ensure txn_date exists
    if 'txn_date' in df.columns:
        df['txn_date_int'] = pd.to_numeric(df['txn_date'], errors='coerce')
    else:
        df['txn_date_int'] = np.nan
    if 'txn_time' not in df.columns:
        df['txn_time'] = ''
    df['txn_time'] = df['txn_time'].replace('', '00:00:00')
    # convert time to timedelta
    def safe_time_to_timedelta(t):
        try:
            return pd.to_timedelta(t)
        except:
            return pd.to_timedelta('00:00:00')
    df['time_only'] = df['txn_time'].apply(safe_time_to_timedelta)
    df['dt'] = df['txn_date_int'].apply(
        lambda x: (base + pd.Timedelta(days=int(x)-1)) if not pd.isna(x) else pd.NaT
    )
    df['dt'] = df['dt'] + df['time_only'].fillna(pd.Timedelta(seconds=0))
    return df

tx_alert = build_dt_from_date_time(tx_alert)
tx_normal = build_dt_from_date_time(tx_normal)
print("alert tx dt 非 NaT:", tx_alert['dt'].notna().sum(), "/", len(tx_alert))
print("normal tx dt 非 NaT:", tx_normal['dt'].notna().sum(), "/", len(tx_normal))

# 若 dt 仍為 NaT 過多，顯示幾筆供檢查
if tx_alert['dt'].notna().sum() < len(tx_alert):
    print("[warn] alert_tx 有些 dt 解析成 NaT，顯示前 5 筆原始 txn_date/txn_time：")
    display(tx_alert[['txn_date','txn_time']].head(10))
if tx_normal['dt'].notna().sum() < len(tx_normal):
    print("[warn] normal_tx 有些 dt 解析成 NaT，顯示前 5 筆原始 txn_date/txn_time：")
    display(tx_normal[['txn_date','txn_time']].head(10))

# ------------- 5) 建 cutoff map -------------
# positive cutoff from acct_alert_df.event_date_dt
pos_cutoff_map = { str(r['acct']): r['event_date_dt'] for _, r in acct_alert_df.iterrows() }

# negative cutoff strategy: sample_from_pos if pos_dates exist, else use last txn
pos_dates = [d for d in acct_alert_df['event_date_dt'].dropna().tolist()]
neg_cutoff_map = {}
neg_accts = acct_normal_df['acct'].astype(str).tolist()
if len(pos_dates) > 0:
    for acct in neg_accts:
        neg_cutoff_map[str(acct)] = random.choice(pos_dates)
else:
    tmp_from = tx_normal[['from_acct','dt']].rename(columns={'from_acct':'acct'})
    tmp_to   = tx_normal[['to_acct','dt']].rename(columns={'to_acct':'acct'})
    tmp_all = pd.concat([tmp_from, tmp_to], ignore_index=True)
    grp_last = tmp_all.groupby('acct')['dt'].max().to_dict()
    for acct in neg_accts:
        neg_cutoff_map[str(acct)] = grp_last.get(acct, pd.NaT)

print("pos_cutoff_map size:", len(pos_cutoff_map), "neg_cutoff_map size:", len(neg_cutoff_map))

# ------------- 6) 建 tx_all 並確保 txn_amt numeric -------------
tx_all = pd.concat([tx_alert, tx_normal], ignore_index=True).fillna("")
if 'txn_amt' in tx_all.columns:
    tx_all['txn_amt'] = pd.to_numeric(tx_all['txn_amt'], errors='coerce').fillna(0.0)

# 儲存到 kernel 供下一 cell 使用
acct_alert = acct_alert_df
alert_tx = tx_alert
normal_tx = tx_normal
acct_normal = acct_normal_df

print("已建立變數：acct_alert, alert_tx, normal_tx, acct_normal, tx_all, pos_cutoff_map, neg_cutoff_map")
print("你現在可以執行先前的 cutoff-aware 特徵化 / 訓練 cell（或我可直接在此 cell 內繼續做訓練）")

[info] 從檔案讀取 acct_alert.csv
[info] 從檔案讀取 alert_transaction.csv
[info] 從檔案讀取 normal_transaction.csv
[info] 從檔案讀取 acct_normal.csv
acct_alert 欄位： ['acct', 'event_date']
alert_tx 欄位（前20）： ['from_acct', 'from_acct_type', 'to_acct', 'to_acct_type', 'is_self_txn', 'txn_amt', 'txn_date', 'txn_time', 'currency_type', 'channel_type']
normal_tx 欄位（前20）： ['from_acct', 'from_acct_type', 'to_acct', 'to_acct_type', 'is_self_txn', 'txn_amt', 'txn_date', 'txn_time', 'currency_type', 'channel_type']
parsed event_date_dt 非 NaT: 1004 / 1004
alert tx dt 非 NaT: 33933 / 33933
normal tx dt 非 NaT: 4367 / 4367
pos_cutoff_map size: 1004 neg_cutoff_map size: 1004
已建立變數：acct_alert, alert_tx, normal_tx, acct_normal, tx_all, pos_cutoff_map, neg_cutoff_map
你現在可以執行先前的 cutoff-aware 特徵化 / 訓練 cell（或我可直接在此 cell 內繼續做訓練）


In [5]:
# 執行前請先確認你已在 kernel 執行過「建立變數並解析 dt」的 cell，
# 並且以下變數已存在：acct_alert, alert_tx (或 tx_alert), normal_tx, tx_all, pos_cutoff_map, neg_cutoff_map
import pandas as pd
import numpy as np
import random
import joblib
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report

# tqdm 容錯載入（Notebook-friendly）
try:
    from tqdm.auto import tqdm
except Exception:
    try:
        from tqdm import tqdm
    except Exception:
        def tqdm(iterable, *args, **kwargs):
            return iterable

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 檢查必要變數
needed = ['acct_alert','tx_all','pos_cutoff_map','neg_cutoff_map']
miss = [v for v in needed if v not in globals()]
if miss:
    raise RuntimeError(f"找不到必要變數: {miss}。請先執行前一個建立變數的 cell。")

acct_alert = globals()['acct_alert']
tx_all = globals()['tx_all']
pos_cutoff_map = globals()['pos_cutoff_map']
neg_cutoff_map = globals()['neg_cutoff_map']

# 穩健的 fallback 特徵函式（若你有更完整的 acct_features，會優先使用）
def fallback_acct_features(df_acct, acct_id=None):
    feat = {}
    feat['total_txns'] = len(df_acct)
    if len(df_acct) == 0:
        feat.update({'in_txns':0,'out_txns':0,'sum_in_amt':0.0,'sum_out_amt':0.0,'max_txn_amt':0.0,'unique_counterparties':0})
        return feat
    if acct_id is not None:
        feat['in_txns'] = int((df_acct['to_acct']==acct_id).sum()) if 'to_acct' in df_acct.columns else 0
        feat['out_txns'] = int((df_acct['from_acct']==acct_id).sum()) if 'from_acct' in df_acct.columns else 0
        feat['sum_in_amt'] = float(df_acct.loc[df_acct['to_acct']==acct_id, 'txn_amt'].astype(float).sum()) if 'txn_amt' in df_acct.columns else 0.0
        feat['sum_out_amt'] = float(df_acct.loc[df_acct['from_acct']==acct_id, 'txn_amt'].astype(float).sum()) if 'txn_amt' in df_acct.columns else 0.0
    else:
        feat['in_txns'] = int((df_acct['to_acct'].notna()).sum()) if 'to_acct' in df_acct.columns else 0
        feat['out_txns'] = int((df_acct['from_acct'].notna()).sum()) if 'from_acct' in df_acct.columns else 0
        feat['sum_in_amt'] = float(df_acct['txn_amt'].astype(float).sum()) if 'txn_amt' in df_acct.columns else 0.0
        feat['sum_out_amt'] = float(df_acct['txn_amt'].astype(float).sum()) if 'txn_amt' in df_acct.columns else 0.0
    feat['max_txn_amt'] = float(df_acct['txn_amt'].astype(float).max()) if ('txn_amt' in df_acct.columns and len(df_acct)>0) else 0.0
    cps = []
    if 'from_acct' in df_acct.columns:
        cps.extend(df_acct['from_acct'].astype(str).tolist())
    if 'to_acct' in df_acct.columns:
        cps.extend(df_acct['to_acct'].astype(str).tolist())
    cps = set([c for c in cps if c and c != acct_id])
    feat['unique_counterparties'] = len(cps)
    return feat

# 計算每個帳戶特徵（容錯：優先呼叫 kernel 中的 acct_features 函式）
def compute_features_per_account_safe(acct_list, tx_df, cutoff_map=None):
    rows = []
    for acct in tqdm(acct_list, desc="計算特徵"):
        acct = str(acct)
        df_acct = tx_df[(tx_df.get('from_acct','')==acct) | (tx_df.get('to_acct','')==acct)].copy()
        if cutoff_map and acct in cutoff_map and pd.notna(cutoff_map[acct]):
            cutoff = pd.to_datetime(cutoff_map[acct])
            df_acct = df_acct[df_acct['dt'] <= cutoff]
        if 'acct_features' in globals():
            try:
                feats = acct_features(df_acct)
            except Exception as e:
                # 若 acct_features 失敗，使用 fallback
                print(f"[warn] acct_features failed for acct={acct}, using fallback. Error: {e}")
                feats = fallback_acct_features(df_acct, acct_id=acct)
        else:
            feats = fallback_acct_features(df_acct, acct_id=acct)
        if hasattr(feats, 'to_dict') and not isinstance(feats, dict):
            feats = feats.to_dict()
        feats['acct'] = acct
        rows.append(feats)
    return pd.DataFrame(rows).fillna(0)

# 取得 pos / neg 帳戶清單
pos_accts = acct_alert['acct'].astype(str).unique().tolist()
neg_accts = pd.read_csv("acct_normal.csv", dtype=str).fillna("")['acct'].astype(str).unique().tolist()

print("pos count:", len(pos_accts), "neg count:", len(neg_accts))

# 計算特徵（這可能需要幾分鐘）
pos_feat_df = compute_features_per_account_safe(pos_accts, tx_all, cutoff_map=pos_cutoff_map)
neg_feat_df = compute_features_per_account_safe(neg_accts, tx_all, cutoff_map=neg_cutoff_map)

pos_feat_df['label'] = 1
neg_feat_df['label'] = 0

train_df = pd.concat([pos_feat_df, neg_feat_df], ignore_index=True).fillna(0)
train_df.to_csv("train_features_cutoffed.csv", index=False)
print("train_features_cutoffed.csv 已寫出，shape=", train_df.shape)

# 訓練 baseline
feature_cols = [c for c in train_df.columns if c not in {'acct','label'}]
X = train_df[feature_cols]
y = train_df['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)
clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)
clf.fit(X_train, y_train)
y_prob = clf.predict_proba(X_test)[:,1]
ap = average_precision_score(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob) if y_test.nunique()>1 else None

print("Baseline -> AP:", round(ap,4), "ROC AUC:", auc)
print(classification_report(y_test, (y_prob>=0.5).astype(int)))

# 儲存模型
joblib.dump(clf, "rf_cutoff_aware.joblib")
print("模型已儲存為 rf_cutoff_aware.joblib")

pos count: 1004 neg count: 1004


計算特徵:   0%|          | 0/1004 [00:00<?, ?it/s]

計算特徵:   0%|          | 0/1004 [00:00<?, ?it/s]

train_features_cutoffed.csv 已寫出，shape= (2008, 9)
Baseline -> AP: 0.9265 ROC AUC: 0.9293829360659389
              precision    recall  f1-score   support

           0       0.88      0.89      0.89       201
           1       0.89      0.88      0.89       201

    accuracy                           0.89       402
   macro avg       0.89      0.89      0.89       402
weighted avg       0.89      0.89      0.89       402

模型已儲存為 rf_cutoff_aware.joblib


# 這段cell執行會出現錯誤訊息，僅儲存參考不執行
# Cell: cutoff-aware 特徵化 + baseline 訓練（假設前一 cell 已建立 acct_alert, alert_tx, normal_tx 並解析好 dt）
import pandas as pd
import numpy as np
import random
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, classification_report
import joblib

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# 1) 建 cutoff map for positives (use parsed event_date_dt)
pos_cutoff_map = {}
if 'event_date_dt' in acct_alert.columns:
    for _, r in acct_alert.iterrows():
        acct = str(r['acct'])
        pos_cutoff_map[acct] = pd.to_datetime(r['event_date_dt']) if pd.notna(r['event_date_dt']) else pd.NaT
else:
    raise RuntimeError("acct_alert 中沒有 event_date_dt，請先執行前一 cell 並確認 event_date 解析成功。")

# 2) 負例 cutoff 策略：用 sample_from_pos（從正例 event_date 分布抽樣），避免時間偏差
pos_dates = [d for d in acct_alert['event_date_dt'].dropna().tolist()]
neg_cutoff_map = {}
neg_accts = pd.read_csv("acct_normal.csv", dtype=str).fillna("")['acct'].astype(str).tolist()
if len(pos_dates) > 0:
    for acct in neg_accts:
        neg_cutoff_map[acct] = random.choice(pos_dates)
else:
    # fallback: 用該負例在 normal_tx 的最後交易時間
    tmp_from = normal_tx[['from_acct','dt']].rename(columns={'from_acct':'acct'})
    tmp_to   = normal_tx[['to_acct','dt']].rename(columns={'to_acct':'acct'})
    tmp_all = pd.concat([tmp_from, tmp_to], ignore_index=True)
    grp_last = tmp_all.groupby('acct')['dt'].max().to_dict()
    for acct in neg_accts:
        neg_cutoff_map[acct] = grp_last.get(acct, pd.NaT)

# 3) 把 alert_tx 與 normal_tx 合併成一個 tx_all（方便從中切割）
tx_all = pd.concat([alert_tx, normal_tx], ignore_index=True).fillna("")
# 確保 txn_amt 為數值
if 'txn_amt' in tx_all.columns:
    tx_all['txn_amt'] = pd.to_numeric(tx_all['txn_amt'], errors='coerce').fillna(0.0)

# 4) 特徵計算：若你 notebook 已有 acct_features 函式會優先使用；否則用簡單 fallback
def fallback_acct_features(df_acct):
    feat = {}
    feat['total_txns'] = len(df_acct)
    feat['in_txns'] = int((df_acct['to_acct'] == df_acct.iloc[0]['acct']).sum()) if len(df_acct)>0 else 0
    feat['out_txns'] = int((df_acct['from_acct'] == df_acct.iloc[0]['acct']).sum()) if len(df_acct)>0 else 0
    feat['sum_in_amt'] = float(df_acct[df_acct['to_acct']==df_acct.iloc[0]['acct']]['txn_amt'].sum()) if len(df_acct)>0 else 0.0
    feat['sum_out_amt'] = float(df_acct[df_acct['from_acct']==df_acct.iloc[0]['acct']]['txn_amt'].sum()) if len(df_acct)>0 else 0.0
    feat['max_txn_amt'] = float(df_acct['txn_amt'].max()) if len(df_acct)>0 else 0.0
    feat['unique_counterparties'] = int(pd.unique(pd.concat([df_acct['from_acct'], df_acct['to_acct']])).size) if len(df_acct)>0 else 0
    return feat

def compute_features_per_account(acct_list, tx_df, cutoff_map=None):
    rows = []
    for acct in tqdm(acct_list, desc="計算特徵"):
        acct = str(acct)
        df_acct = tx_df[(tx_df['from_acct']==acct) | (tx_df['to_acct']==acct)].copy()
        if cutoff_map and acct in cutoff_map and pd.notna(cutoff_map[acct]):
            cutoff = pd.to_datetime(cutoff_map[acct])
            df_acct = df_acct[df_acct['dt'] <= cutoff]
        # preferentially call existing acct_features if available
        if 'acct_features' in globals():
            try:
                feats = acct_features(df_acct)
            except Exception:
                feats = fallback_acct_features(df_acct)
        else:
            feats = fallback_acct_features(df_acct)
        feats['acct'] = acct
        rows.append(feats)
    return pd.DataFrame(rows).fillna(0)

# 5) 計算正負例特徵（會花一些時間）
pos_accts = acct_alert['acct'].astype(str).unique().tolist()
print("pos count:", len(pos_accts), "neg count:", len(neg_accts))
pos_feat_df = compute_features_per_account(pos_accts, tx_all, cutoff_map=pos_cutoff_map)
neg_feat_df = compute_features_per_account(neg_accts, tx_all, cutoff_map=neg_cutoff_map)
pos_feat_df['label'] = 1
neg_feat_df['label'] = 0

train_df = pd.concat([pos_feat_df, neg_feat_df], ignore_index=True).fillna(0)
train_df.to_csv("train_features_cutoffed.csv", index=False)
print("已儲存 train_features_cutoffed.csv, shape:", train_df.shape)

# 6) 簡單 train/test split 與 RandomForest baseline
feature_cols = [c for c in train_df.columns if c not in {'acct','label'}]
X = train_df[feature_cols]
y = train_df['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_SEED)
clf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)
clf.fit(X_train, y_train)
y_prob = clf.predict_proba(X_test)[:,1]
ap = average_precision_score(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob) if y_test.nunique()>1 else None
print("AP:", ap, "ROC AUC:", auc)
print(classification_report(y_test, (y_prob>=0.5).astype(int)))

# 儲存模型
joblib.dump(clf, "rf_cutoff_aware.joblib")
print("已儲存模型 rf_cutoff_aware.joblib")